# Flujo completo: carga, transformación y resumen de `solomonk/reddit_mental_health_posts`

Este notebook coordina las dos clases del proyecto:

```
HFDatasetLoader → DataFrame → transformación mínima
                            → DatasetSummarizer → archivo .txt
```

1. Importar las dos clases desde sus archivos `.py`.
2. Obtener el DataFrame desde Hugging Face con `HFDatasetLoader`.
3. Aplicar una transformación mínima al DataFrame.
4. Pasar el DataFrame transformado a `DatasetSummarizer` para generar el `.txt`.
5. Mostrar el resultado y explicar la transformación.

Las clases contienen la lógica de carga y de resumen; el notebook solo coordina el flujo y hace la transformación.

## 1. Importar las clases

Se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Luego se importa cada clase desde su propio archivo:

- `hf_loader.py` → `HFDatasetLoader` (conexión a Hugging Face)
- `dataset_summarizer.py` → `DatasetSummarizer` (resumen en `.txt`)

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po.hf_loader import HFDatasetLoader
from clases_po.dataset_summarizer import DatasetSummarizer

print("Raíz del proyecto:", ROOT)

Raíz del proyecto: C:\Users\aaram\Documents\Primer semestre Ingenierria de software\Analitica de datos\Act1\Analitica20262


## 2. Cargar el dataset con la primera clase

`HFDatasetLoader` descarga el split `train` y lo devuelve como `pandas.DataFrame`. El notebook no se conecta a Hugging Face directamente: esa responsabilidad es de la clase.

In [2]:
loader = HFDatasetLoader(
    dataset_name="solomonk/reddit_mental_health_posts",
    split="train",
)

df = loader.load()

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print("Nombres:", list(df.columns))
df.head()

Repo card metadata block was not found. Setting CardData to empty.


Filas: 151,288
Columnas: 10
Nombres: ['author', 'body', 'created_utc', 'id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', 'url']


,author,body,created_utc,id,num_comments,score,subreddit,title,upvote_ratio,url
0,HotConversation1273,A few months ago I was accepted into this full...,2021-12-22T18:32:56.000Z,rmbjwb,1,1,ADHD,I get extremely anxious if I’m not working 24/7,1.0,https://www.reddit.com/r/ADHD/comments/rmbjwb/...
1,snorefestt,"Hey guys, I was curious if anyone else has the...",2021-12-22T18:24:25.000Z,rmbd1y,3,5,ADHD,"I can't will myself to clean my own house, but...",1.0,https://www.reddit.com/r/ADHD/comments/rmbd1y/...
2,etyf12,\n\ni have 6 exams in the next 2 weeks one of...,2021-12-22T18:22:52.000Z,rmbbvu,1,2,ADHD,i need some help,1.0,https://www.reddit.com/r/ADHD/comments/rmbbvu/...
3,GetHairOrDieTryin,Is there anyone out there that is struggling w...,2021-12-22T18:20:35.000Z,rmba1t,3,2,ADHD,Anyone up for a chat?,1.0,https://www.reddit.com/r/ADHD/comments/rmba1t/...
4,ZeroTransPat,"Whenever I get hungry, I never eat because I d...",2021-12-22T18:18:47.000Z,rmb8lm,2,1,ADHD,Figuring out what to eat sucks,1.0,https://www.reddit.com/r/ADHD/comments/rmb8lm/...


## 3. Transformación mínima

En Reddit, muchos posts no tienen contenido real en `body`: el texto está vacío o fue reemplazado por `[removed]` (eliminado por moderadores) o `[deleted]` (borrado por el autor). Estas filas **no aparecen como nulos**, pero no aportan texto para analizar.

**Transformación:** filtrar las filas y conservar solo los posts cuyo `body` tiene contenido real, es decir:
- `body` no es nulo,
- `body` no está vacío ni contiene solo espacios,
- `body` no es `[removed]` ni `[deleted]`.

El filtro devuelve un **nuevo DataFrame**, que se guarda en `df_transformado`. Ese es el que se pasa a la clase de resumen.

In [3]:
marcadores = ["[removed]", "[deleted]"]
cuerpo = df["body"].fillna("").str.strip()

df_transformado = df[(cuerpo != "") & (~cuerpo.isin(marcadores))].reset_index(drop=True)

print(f"Filas antes:   {len(df):,}")
print(f"Filas después: {len(df_transformado):,}")
print(f"Filas eliminadas: {len(df) - len(df_transformado):,} "
      f"({(len(df) - len(df_transformado)) / len(df):.1%})")

Filas antes:   151,288
Filas después: 87,078
Filas eliminadas: 64,210 (42.4%)


Comparación del contenido de `body` antes y después del filtro:

In [4]:
import pandas as pd

def tipo_body(serie):
    """Clasifica cada valor de body en: con contenido, vacío, [removed] o [deleted]."""
    s = serie.fillna("").str.strip()
    categoria = pd.Series("con contenido", index=s.index)
    categoria[s == ""] = "(vacío o nulo)"
    categoria[s == "[removed]"] = "[removed]"
    categoria[s == "[deleted]"] = "[deleted]"
    return categoria.value_counts()

comparacion = pd.DataFrame({
    "antes": tipo_body(df["body"]),
    "después": tipo_body(df_transformado["body"]),
}).fillna(0).astype(int)
comparacion

,antes,después
(vacío o nulo),1609,0
[deleted],38575,0
[removed],24026,0
con contenido,87078,87078


In [5]:
df_transformado.head()

,author,body,created_utc,id,num_comments,score,subreddit,title,upvote_ratio,url
0,HotConversation1273,A few months ago I was accepted into this full...,2021-12-22T18:32:56.000Z,rmbjwb,1,1,ADHD,I get extremely anxious if I’m not working 24/7,1.0,https://www.reddit.com/r/ADHD/comments/rmbjwb/...
1,snorefestt,"Hey guys, I was curious if anyone else has the...",2021-12-22T18:24:25.000Z,rmbd1y,3,5,ADHD,"I can't will myself to clean my own house, but...",1.0,https://www.reddit.com/r/ADHD/comments/rmbd1y/...
2,etyf12,\n\ni have 6 exams in the next 2 weeks one of...,2021-12-22T18:22:52.000Z,rmbbvu,1,2,ADHD,i need some help,1.0,https://www.reddit.com/r/ADHD/comments/rmbbvu/...
3,GetHairOrDieTryin,Is there anyone out there that is struggling w...,2021-12-22T18:20:35.000Z,rmba1t,3,2,ADHD,Anyone up for a chat?,1.0,https://www.reddit.com/r/ADHD/comments/rmba1t/...
4,ZeroTransPat,"Whenever I get hungry, I never eat because I d...",2021-12-22T18:18:47.000Z,rmb8lm,2,1,ADHD,Figuring out what to eat sucks,1.0,https://www.reddit.com/r/ADHD/comments/rmb8lm/...


## 4. Generar el resumen con la segunda clase

Se pasa **`df_transformado`** (no `df`) a `DatasetSummarizer`, de modo que el archivo `.txt` describe los datos ya filtrados.

In [6]:
resumidor = DatasetSummarizer()

ruta = resumidor.guardar_resumen(
    df_transformado,
    "reportes/reddit_mental_health_posts_filtrado.txt",
    nombre="solomonk/reddit_mental_health_posts (solo posts con body)",
)

print(f"Resumen guardado en: {ruta.resolve()}")

Resumen guardado en: C:\Users\aaram\Documents\Primer semestre Ingenierria de software\Analitica de datos\Act1\Analitica20262\src\sincronicas\reportes\reddit_mental_health_posts_filtrado.txt


## 5. Resultado

Se muestra el contenido del archivo `.txt` generado para comprobar que corresponde a los datos transformados (el número de filas debe coincidir con `len(df_transformado)` y la sección de valores vacíos ya no debe mostrar `[removed]` ni `[deleted]` en `body`).

In [7]:
print(ruta.read_text(encoding="utf-8"))

RESUMEN DEL DATASET: solomonk/reddit_mental_health_posts (solo posts con body)
Generado: 2026-10-24 21:05:09

1. DIMENSIONES
--------------
Filas:    87,078
Columnas: 10
Memoria aproximada: 101.60 MB

2. COLUMNAS Y TIPOS DE DATOS
----------------------------
     columna    tipo  no_nulos  valores_unicos
      author     str     87078           50750
        body     str     87078           86607
 created_utc     str     87078           86834
          id     str     87078           87078
num_comments   int64     87078             420
       score   int64     87078             983
   subreddit     str     87078               5
       title     str     87078           83648
upvote_ratio float64     87078              82
         url     str     87078           87078

3. VALORES NULOS Y VACÍOS
-------------------------
No hay valores nulos (NaN/None) en ninguna columna.

Valores vacíos o de relleno (no cuentan como nulos, pero no tienen contenido útil):
columna  cadenas_vacias  [removed]

## Conclusión

- **Carga:** `HFDatasetLoader` obtuvo el dataset desde Hugging Face como DataFrame.
- **Transformación:** se filtraron las filas cuyo `body` estaba vacío o era `[removed]` / `[deleted]`. Así el conjunto queda solo con publicaciones que tienen texto real, que son las útiles para analizar el contenido de los posts sobre salud mental. El resultado se guardó en un nuevo DataFrame, `df_transformado`.
- **Resumen:** `DatasetSummarizer` recibió `df_transformado` y generó el archivo `reportes/reddit_mental_health_posts_filtrado.txt`, que describe los datos ya transformados.